# Recursive Prompt Outline

## Inputs:
- Entry Class
- abstract token list (JSON list of dicts)
- Required Property map:
```
{
    class: {
        property: (validator, description)
    }
}
```
- Optional Property map (same structure as required property map)

## Output:
Recursively nested JSON dictionary

## Recursive Steps:
1. Find required and optional properties mapped to entry class
2. Cluster tokens to properties based on property description. Do not proceed to step 3 until all tokens have been clustered.
   1. Rules:
      1. All required properties must be complete in output
      2. All tokens must be clustered to a property
      3. Quantities and their respective units should be clustered to two separate properties
      4. Additional properties may be hallucinated to accomodate tokens without a clear property to cluster
      5. Hallucinated properties must be accompanied by a description and should be appended to the properties mapped to the class context in the optional property map at time of creation.
      6. A token cannot be clustered to multiple properties.
   2. Priorities (in decreasing order of importance)
      1. Avoid clustering quantity values or units with other tokens.
      2. If an expected property is present with a description implying flexible context, such as observations, applicable non-quantity-related tokens should be clustered to that property before property hallucination
      3. Avoid hallucinating properties
3. For each property:
   1. dispatch:
      1. If validator is a class in property map, dispatch returns output from recursion to step 1 with:
         1. entry class = validator
         2. abstract token list = clustered property list
         3. property map = property map
      2. Otherwise, dispatch returns tokens formatted as string to match description
   2. property is mapped to dispatch return value



# First Prompt Attempt

In [ ]:
import json

def get_refined_recursive_prompt(
    entry_class, abstract_token_list, required_map, optional_map
):
    return f"""
    You are an algorithmic semantic compiler. Your objective is to compile an
    unstructured abstract token list into a recursively nested JSON dictionary
    tree using the provided rules, priorities, and structural schema maps.

    ### INPUT DATA CONTEXT
    - TARGET ROOT ENTRY CLASS: {json.dumps(entry_class)}
    - ABSTRACT TOKEN LIST: 
    \"\"\" 
    {json.dumps(abstract_token_list, indent=2)}
    \"\"\" 
    - REQUIRED PROPERTY MAP SCHEMA: 
    \"\"\" 
    {json.dumps(required_map, indent=2)} 
    \"\"\" 
    - OPTIONAL PROPERTY MAP SCHEMA: 
    \"\"\"
    {json.dumps(optional_map, indent=2)} 
    \"\"\"

    ### MANDATORY COMPILATION RECURSIVE STEPS 
    Execute these steps chronologically. You must maintain an execution log tracking your state,
    depth, and constraint evaluations.

    STEP 1: PROP LOOKUP 
    Identify all required and optional properties mapped to the current operational entry class context.

    STEP 2: TOKEN CLUSTERING (DO NOT PROCEED TO STEP 3 UNTIL COMPLETE)
    Systematically evaluate and assign EVERY token to exactly one property. You
    must rigidly respect the following rules and decreasing order of priority:
    
    [PRIORITY 1]: Always do all that is possible to complete a required property within the boundaries of the critical clustering laws.
    [PRIORITY 2]: Never cluster numerical quantity values or their unit tokens
    together with other abstract facts. They must be split into two separate
    target properties. 
    [PRIORITY 3]: Check for flexible-context catch-all properties mapped to the class 
    (e.g., 'observations', 'notes', 'comments'). If present, allocate non-quantity text tokens 
    here first before generating new keys. 
    [PRIORITY 4]: Avoid generating new keys. If a token cannot fit anywhere else, you may 
    hallucinate a new property key. Append this new key, along with a string validator and a 
    written functional description, to the context class inside your Optional Property Map memory.
    
    [CRITICAL CLUSTERING LAWS]: 
    - All required properties for the class context must be complete in the output matrix or logged in the output. 
    - Zero tokens can be left unclustered. 
    - A single token cannot map to multiple properties.
    - The required map cannot be changed.
    - Never create new tokens.

    STEP 3: EVALUATION DISPATCH 
    For each identified or hallucinated property inside the current layer, evaluate its validation criteria: 
    - IF VALIDATOR IS A RECOGNIZED CLASS TYPE: 
      Increment recursive depth. Before proceeding to the next property, wait for the return value from a nested
      context framework with shared map memory dispatched back to Step 1 where:
        * Entry Class = Validator Class
        * Abstract Token List = The exact subset of tokens clustered to this specific property
        * Required Map = The exact required map in your memory
        * Optional Map = The exact optional map in your current memory
    - IF VALIDATOR IS A PRIMITIVE/STRING TYPE OR UNRECOGNIZED: 
      Format the clustered sub-tokens into a single cohesive string layout that satisfies the property description.
    
    Map the current key to this dispatched return value.

    ### OUTPUT SPECIFICATION 
    You must format your generation output using this exact structure:

    --- FAILED REQUIRED PROPERTY LOG ---
    Provide a summary of all required properties that were not completed, including the rejected list of available tokens for each failed property.
    --- END FAILED REQUIRED PROPERTY LOG ---

    --- START RUNTIME LOG --- 
    Provide a step-by-step trace of your recursive tree traversal, depth status, token allocations, and any added hallucinated property descriptions. 
    --- END RUNTIME LOG ---

    --- START COMPILED DATA TREE --- 
    Output ONLY a valid, recursively nested JSON object representing the final tree resolution. No preambles, no trailing text. 
    --- END COMPILED DATA TREE ---
    """
